<a href="https://colab.research.google.com/github/moyan-li/MSSP607/blob/main/WeeklyModules/Week09/Week9participation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
import os

github_token = userdata.get('Key1')
owner = 'moyan-li' # Replace with the GitHub repository owner
repository = 'MSSP607' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directory to: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Changed directory to: /content/MSSP607


In [20]:
%pip install requests
import os
import time
import requests

In [21]:

import csv
import os
import time
from urllib.parse import urlparse
import requests


def download_images(keyword, folder="commons_images", max_images=None):
    """Use None for all accessible results, or a positive download limit."""
    if not keyword.strip():
        raise ValueError("Please enter a search keyword.")
    os.makedirs(folder, exist_ok=True)
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Week09CommonsDownloader/1.0 (educational Python exercise)"
    })
    params = {
        "action": "query", "format": "json",
        "generator": "search", "gsrsearch": keyword,
        "gsrnamespace": 6, "gsrlimit": 20,
        "prop": "imageinfo", "iiprop": "url|mime",
    }
    seen = set()
    saved = failed = 0
    extensions = {"image/jpeg": ".jpg", "image/png": ".png",
                  "image/gif": ".gif", "image/webp": ".webp"}
    # Keep source-page links so image credits and licenses can be checked.
    with open(os.path.join(folder, "sources.csv"), "w", newline="", encoding="utf-8") as log:
        writer = csv.writer(log)
        writer.writerow(["filename", "title", "source_page", "image_url"])
        while True:
            response = session.get("https://commons.wikimedia.org/w/api.php",
                                   params=params, timeout=30)
            response.raise_for_status()
            result = response.json()
            if "error" in result:
                raise RuntimeError(result["error"].get("info", "Search failed."))
            pages = result.get("query", {}).get("pages", {})
            for page in pages.values():
                page_id = page["pageid"]
                if page_id in seen:
                    continue
                seen.add(page_id)
                info = page.get("imageinfo", [{}])[0]
                extension = extensions.get(info.get("mime"))
                if not extension or not info.get("url"):
                    continue  # Skip videos, documents, and unsupported formats.
                filename = str(page_id) + extension
                destination = os.path.join(folder, filename)
                temporary = destination + ".part"
                try:
                    with session.get(info["url"], stream=True, timeout=60) as image:
                        image.raise_for_status()
                        if not image.headers.get("Content-Type", "").startswith("image/"):
                            raise ValueError("The server did not return an image.")
                        with open(temporary, "wb") as image_file:
                            for chunk in image.iter_content(100000):
                                if chunk:
                                    image_file.write(chunk)
                    os.replace(temporary, destination)
                    writer.writerow([filename, page["title"],
                                     info.get("descriptionurl", ""), info["url"]])
                    log.flush()
                    saved += 1
                    print("Downloaded:", filename)
                except (requests.RequestException, OSError, ValueError) as error:
                    failed += 1
                    if os.path.exists(temporary):
                        os.remove(temporary)
                    print("Download failed:", page["title"], error)
                time.sleep(1)
                if max_images is not None and saved >= max_images:
                    print("Reached the selected limit:", saved)
                    return saved, failed
            if "continue" not in result:
                break
            params.update(result["continue"])
            time.sleep(1)
    print("Finished. Downloaded:", saved, "Failed:", failed)
    print("Folder:", os.path.abspath(folder))
    return saved, failed




In [22]:
keyword = input("Enter an image category (for example, cats): ").strip()

try:
    download_images(keyword, max_images=None)
except (requests.RequestException, RuntimeError, ValueError) as error:
    print("Stopped:", error)


Enter an image category (for example, cats): amazon
Downloaded: 120118733.jpg
Downloaded: 183641183.png
Downloaded: 32011486.jpg
Downloaded: 159794261.jpg
Downloaded: 147560344.png
Reached the selected limit: 5
